# Flight Dynamics
*From aerodynamic forces to stability modes — with live simulations*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider
from IPython.display import display, clear_output
from scipy.integrate import solve_ivp

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

G0 = 9.80665        # m/s^2
RHO_SL = 1.225      # kg/m^3 sea level
T_SL = 288.15       # K sea level
P_SL = 101325       # Pa
LAPSE = 0.0065      # K/m troposphere lapse rate
R_AIR = 287.058     # J/(kg·K)

def isa(h):
    """ISA atmosphere. Returns (T, P, rho) at altitude h [m]."""
    h = np.asarray(h, dtype=float)
    T = np.where(h <= 11000, T_SL - LAPSE * h, 216.65)
    P = np.where(h <= 11000,
                 P_SL * (T / T_SL) ** (G0 / (LAPSE * R_AIR)),
                 22632 * np.exp(-G0 * (h - 11000) / (R_AIR * 216.65)))
    rho = P / (R_AIR * T)
    return T, P, rho

rows = [
    ('ρ₀ (sea level)',  f'{RHO_SL:.3f}',     'kg/m³'),
    ('T₀ (sea level)',  f'{T_SL:.2f}',        'K'),
    ('P₀ (sea level)',  f'{P_SL:.0f}',        'Pa'),
    ('Lapse rate',      f'{LAPSE*1e3:.1f}',   '°C/km'),
    ('Tropopause',      '11 000',             'm'),
    ('g₀',              f'{G0:.5f}',          'm/s²'),
]
print(f"{'Parameter':<20} {'Value':>12}  Unit")
print('-' * 42)
for name, val, unit in rows:
    print(f'{name:<20} {val:>12}  {unit}')

Parameter                   Value  Unit
------------------------------------------
ρ₀ (sea level)              1.225  kg/m³
T₀ (sea level)             288.15  K
P₀ (sea level)             101325  Pa
Lapse rate                    6.5  °C/km
Tropopause                 11 000  m
g₀                        9.80665  m/s²


## 1 — Aerodynamic Forces: Lift and Drag

$$L = \tfrac{1}{2}\rho V^2 S\, C_L \qquad D = \tfrac{1}{2}\rho V^2 S\, C_D$$

| Symbol | Meaning |
|:--|:--|
| $\rho$ | Air density (decreases with altitude) |
| $V$ | True airspeed |
| $S$ | Wing reference area |
| $C_L, C_D$ | Lift and drag coefficients (functions of angle of attack $\alpha$) |

In level flight: $L = W$, giving the **stall speed** (minimum speed at $C_{L_{\max}}$):

$$V_{\text{stall}} = \sqrt{\frac{2W}{\rho\,S\,C_{L_{\max}}}}$$

In [2]:
out1 = widgets.Output()

alt1_sl = FloatSlider(value=0, min=0, max=12000, step=500,
    description='Altitude (m)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
mass_sl = FloatSlider(value=75000, min=5000, max=300000, step=5000,
    description='Mass (kg)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
S_sl = FloatSlider(value=122.6, min=10, max=500, step=5,
    description='Wing area (m²)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
clmax_sl = FloatSlider(value=2.0, min=1.0, max=3.5, step=0.1,
    description='CL_max', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_forces(change=None):
    h = alt1_sl.value
    m = mass_sl.value
    S = S_sl.value
    CLmax = clmax_sl.value
    W = m * G0
    _, _, rho = isa(h)
    V = np.linspace(30, 300, 500)
    Vs = np.sqrt(2 * W / (rho * S * CLmax))
    # CL required for level flight at each V
    CL_req = 2 * W / (rho * V**2 * S)
    CL_req = np.clip(CL_req, 0, CLmax)
    # drag polar: CD = CD0 + CL²/(pi*e*AR)
    CD0 = 0.020
    e_ow = 0.8
    AR = 9.0
    CD = CD0 + CL_req**2 / (np.pi * e_ow * AR)
    L = 0.5 * rho * V**2 * S * CL_req
    D = 0.5 * rho * V**2 * S * CD
    LD = CL_req / CD
    V_minD = V[np.argmin(D[V > Vs] if np.any(V > Vs) else D)]

    with out1:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # force diagram
        ax1.set_xlim(-2, 2)
        ax1.set_ylim(-2, 2)
        ax1.set_aspect('equal')
        ax1.axis('off')
        ax1.set_title('Forces in level flight', fontsize=10)
        # aircraft body
        body = plt.Polygon([[-1.2, -0.05], [1.2, -0.05], [1.2, 0.05], [-1.2, 0.05]],
                           fc='#bdc3c7', ec='k', lw=1.5)
        ax1.add_patch(body)
        wing = plt.Polygon([[-0.4, -0.02], [0.4, -0.02], [0.5, 0.04], [-0.5, 0.04]],
                           fc='#95a5a6', ec='k', lw=1)
        ax1.add_patch(wing)
        # lift arrow (up)
        ax1.annotate('', xy=(0, 1.4), xytext=(0, 0.1),
                     arrowprops=dict(arrowstyle='->', color='C0', lw=3))
        ax1.text(0.15, 0.8, f'L = {W/1e3:.0f} kN', fontsize=11, color='C0', fontweight='bold')
        # weight arrow (down)
        ax1.annotate('', xy=(0, -1.4), xytext=(0, -0.1),
                     arrowprops=dict(arrowstyle='->', color='C3', lw=3))
        ax1.text(0.15, -1.0, f'W = {W/1e3:.0f} kN', fontsize=11, color='C3', fontweight='bold')
        # thrust arrow (right)
        ax1.annotate('', xy=(1.8, 0), xytext=(1.2, 0),
                     arrowprops=dict(arrowstyle='->', color='C2', lw=2.5))
        ax1.text(1.5, 0.15, 'T', fontsize=11, color='C2', fontweight='bold')
        # drag arrow (left)
        ax1.annotate('', xy=(-1.8, 0), xytext=(-1.2, 0),
                     arrowprops=dict(arrowstyle='->', color='C1', lw=2.5))
        ax1.text(-1.7, 0.15, 'D', fontsize=11, color='C1', fontweight='bold')
        ax1.text(0, -1.8, f'V_stall = {Vs:.0f} m/s ({Vs*3.6:.0f} km/h)',
                 ha='center', fontsize=10, color='C3')

        # L, D vs V
        ax2.plot(V * 3.6, L / 1e3, 'C0', lw=2, label='Lift')
        ax2.plot(V * 3.6, D / 1e3, 'C1', lw=2, label='Drag')
        ax2.axhline(W / 1e3, ls=':', color='C3', lw=1, alpha=0.6, label='Weight')
        ax2.axvline(Vs * 3.6, ls='--', color='C3', lw=1.2, alpha=0.6)
        ax2.text(Vs*3.6 + 5, ax2.get_ylim()[1]*0.1 if ax2.get_ylim()[1] > 0 else 10,
                 f'V_stall\n{Vs:.0f} m/s', fontsize=8, color='C3')
        ax2.set_xlabel('True airspeed (km/h)')
        ax2.set_ylabel('Force (kN)')
        ax2.set_title(f'Lift & Drag vs Speed — h={h:.0f} m  ρ={rho:.3f} kg/m³', fontsize=10)
        ax2.legend(fontsize=9)
        ax2.set_xlim(0, 300 * 3.6)
        plt.tight_layout()
        plt.show()

for w in [alt1_sl, mass_sl, S_sl, clmax_sl]:
    w.observe(update_forces, 'value')
display(widgets.VBox([alt1_sl, mass_sl, S_sl, clmax_sl]), out1)
update_forces()

Output()

## 2 — Airfoil Characteristics: CL and CD vs Alpha

Below stall, lift coefficient is approximately linear:

$$C_L \approx C_{L_\alpha}\,(\alpha - \alpha_0), \qquad C_{L_\alpha} \approx 2\pi \text{ (thin airfoil)}$$

The **drag polar** relates lift to drag:

$$C_D = C_{D_0} + \frac{C_L^2}{\pi\, e\, AR}$$

Maximum $L/D$ occurs when parasitic drag equals induced drag:

$$\left(\frac{L}{D}\right)_{\max} = \frac{1}{2}\sqrt{\frac{\pi\, e\, AR}{C_{D_0}}}$$

In [3]:
out2 = widgets.Output()

ar_sl = FloatSlider(value=9.0, min=3, max=30, step=0.5,
    description='Aspect ratio', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
e_sl = FloatSlider(value=0.8, min=0.5, max=0.95, step=0.01,
    description='Oswald e', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
cd0_sl = FloatSlider(value=0.020, min=0.005, max=0.060, step=0.001,
    description='CD0', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False,
    readout_format='.3f')
stall_sl = FloatSlider(value=15, min=8, max=22, step=0.5,
    description='α_stall (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_polar(change=None):
    AR = ar_sl.value
    e_ow = e_sl.value
    CD0 = cd0_sl.value
    alpha_stall = stall_sl.value
    CLa = 2 * np.pi / (1 + 2/AR)  # corrected for finite wing
    alpha = np.linspace(-5, 25, 500)
    alpha_rad = np.deg2rad(alpha)
    # CL vs alpha
    CL_linear = CLa * alpha_rad
    CLmax = CLa * np.deg2rad(alpha_stall)
    CL = np.where(alpha <= alpha_stall,
                  CL_linear,
                  CLmax - 3.0 * (np.deg2rad(alpha - alpha_stall))**1.5)
    CL = np.clip(CL, -0.5, CLmax * 1.05)
    CD = CD0 + CL**2 / (np.pi * e_ow * AR)
    LD = CL / CD
    LD_max = 0.5 * np.sqrt(np.pi * e_ow * AR / CD0)
    CL_best = np.sqrt(CD0 * np.pi * e_ow * AR)
    alpha_best = np.rad2deg(CL_best / CLa)
    CD_best = CD0 + CL_best**2 / (np.pi * e_ow * AR)

    with out2:
        clear_output(wait=True)
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))
        # CL vs alpha
        ax1.plot(alpha, CL, 'C0', lw=2)
        ax1.axvline(alpha_stall, ls='--', color='C3', lw=1, alpha=0.6)
        ax1.fill_between(alpha, CL, where=alpha > alpha_stall, alpha=0.1, color='C3')
        ax1.text(alpha_stall + 0.5, CLmax * 0.5, 'STALL', fontsize=9, color='C3', rotation=90)
        ax1.plot(alpha_best, CL_best, 'o', color='C2', ms=8, zorder=5)
        ax1.annotate(f'Best L/D\nα={alpha_best:.1f}°', (alpha_best, CL_best),
                     xytext=(alpha_best+3, CL_best-0.3), fontsize=8, color='C2',
                     arrowprops=dict(arrowstyle='->', color='C2'))
        ax1.set_xlabel('Angle of attack α (°)')
        ax1.set_ylabel('CL')
        ax1.set_title('Lift curve')
        ax1.set_xlim(-5, 25)

        # drag polar CL vs CD
        mask = alpha <= alpha_stall + 2
        ax2.plot(CD[mask], CL[mask], 'C1', lw=2)
        ax2.plot(CD_best, CL_best, 'o', color='C2', ms=8, zorder=5)
        # tangent line from origin = max L/D
        cd_line = np.linspace(0, CD[mask].max(), 100)
        ax2.plot(cd_line, LD_max * cd_line, 'C2', ls=':', lw=1,
                 label=f'(L/D)_max = {LD_max:.1f}')
        ax2.set_xlabel('CD')
        ax2.set_ylabel('CL')
        ax2.set_title('Drag polar')
        ax2.legend(fontsize=8)

        # L/D vs alpha
        ax3.plot(alpha[alpha <= alpha_stall], LD[alpha <= alpha_stall], 'C2', lw=2)
        ax3.axhline(LD_max, ls=':', color='C2', lw=1, alpha=0.5)
        ax3.plot(alpha_best, LD_max, 'o', color='C2', ms=8, zorder=5)
        ax3.text(alpha_best + 1, LD_max + 0.5, f'{LD_max:.1f}', fontsize=10,
                 color='C2', fontweight='bold')
        ax3.set_xlabel('α (°)')
        ax3.set_ylabel('L/D')
        ax3.set_title('Lift-to-drag ratio')
        ax3.set_xlim(-5, alpha_stall + 1)

        fig.suptitle(f'AR = {AR:.1f}  e = {e_ow:.2f}  CD0 = {CD0:.3f}  '
                     f'CLmax = {CLmax:.2f}  (L/D)max = {LD_max:.1f}',
                     fontsize=10, y=1.02)
        plt.tight_layout()
        plt.show()

for w in [ar_sl, e_sl, cd0_sl, stall_sl]:
    w.observe(update_polar, 'value')
display(widgets.VBox([ar_sl, e_sl, cd0_sl, stall_sl]), out2)
update_polar()

Output()

/var/folders/sn/ms08v0xn03sdzyg1814wfjzc0000gn/T/ipykernel_49029/2547820551.py:30: RuntimeWarning: invalid value encountered in power
  CLmax - 3.0 * (np.deg2rad(alpha - alpha_stall))**1.5)


## 3 — ISA Atmosphere Model

The **International Standard Atmosphere** defines reference profiles for temperature, pressure, and density:

**Troposphere** ($h \le 11\,\text{km}$):
$$T = T_0 - L\,h, \quad P = P_0\left(\frac{T}{T_0}\right)^{g/(LR)}, \quad \rho = \frac{P}{RT}$$

**Stratosphere** ($h > 11\,\text{km}$): $T = 216.65$ K (isothermal), pressure decays exponentially.

| Altitude | Aircraft |
|:--|:--|
| 0–3 km | General aviation (Cessna 172) |
| 8–12 km | Commercial jets (737, A320) |
| 15–20 km | Military (F-15, F-22) |
| 20–26 km | U-2, SR-71 |

In [ ]:
@interact(
    h_max=IntSlider(value=30000, min=5000, max=50000, step=1000,
                    description='Max alt (m)', style={'description_width':'110px'},
                    continuous_update=False)
)
def plot_isa(h_max):
    plt.close('all')
    h = np.linspace(0, h_max, 500)
    T, P, rho = isa(h)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
    # Temperature
    ax1.plot(T - 273.15, h / 1e3, 'C3', lw=2)
    ax1.set_xlabel('Temperature (°C)')
    ax1.set_ylabel('Altitude (km)')
    ax1.set_title('Temperature')
    ax1.axhline(11, ls=':', color='gray', lw=1, alpha=0.5)
    ax1.text(T[0]-273.15, 11.3, 'Tropopause', fontsize=8, color='gray')
    # Pressure
    ax2.plot(P / 1e3, h / 1e3, 'C0', lw=2)
    ax2.set_xlabel('Pressure (kPa)')
    ax2.set_title('Pressure')
    ax2.axhline(11, ls=':', color='gray', lw=1, alpha=0.5)
    # Density
    ax3.plot(rho, h / 1e3, 'C2', lw=2)
    ax3.set_xlabel('Density (kg/m³)')
    ax3.set_title('Density')
    ax3.axhline(11, ls=':', color='gray', lw=1, alpha=0.5)
    # aircraft annotations
    aircraft = [('Cessna 172', 2.5, 'C1'), ('B737/A320', 10.7, 'C0'),
                ('F-22', 18, 'C4'), ('U-2/SR-71', 24, 'C3')]
    for name, alt, c in aircraft:
        if alt <= h_max / 1e3:
            for ax in [ax1, ax2, ax3]:
                ax.axhline(alt, ls='--', color=c, lw=0.8, alpha=0.4)
            ax3.text(rho.max()*0.5, alt+0.3, name, fontsize=8, color=c)
    plt.tight_layout()
    plt.show()

## 4 — Aircraft Performance: Thrust Required vs Available

In level flight, thrust equals drag. The **thrust required** curve has two components:

$$T_R = D = \underbrace{\tfrac{1}{2}\rho V^2 S\,C_{D_0}}_{\text{parasite drag}} + \underbrace{\frac{2W^2}{\rho V^2 S \pi e\, AR}}_{\text{induced drag}}$$

- **Minimum drag speed** $V_{MD}$: where total drag is minimum (parasite = induced)
- **Minimum power speed** $V_{MP} = V_{MD} / 3^{1/4} \approx 0.76\,V_{MD}$
- Maximum speed: where thrust available = thrust required

$$V_{MD} = \sqrt{\frac{2W}{\rho S}\sqrt{\frac{1}{\pi e\,AR\,C_{D_0}}}}$$

In [4]:
out4 = widgets.Output()

W_sl = FloatSlider(value=700000, min=50000, max=2000000, step=50000,
    description='Weight (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
h4_sl = FloatSlider(value=0, min=0, max=12000, step=500,
    description='Altitude (m)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
cd04_sl = FloatSlider(value=0.020, min=0.010, max=0.050, step=0.002,
    description='CD0', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False,
    readout_format='.3f')
ar4_sl = FloatSlider(value=9.0, min=4, max=20, step=0.5,
    description='AR', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
Ta_sl = FloatSlider(value=120000, min=10000, max=500000, step=5000,
    description='T_avail (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_perf(change=None):
    W = W_sl.value
    h = h4_sl.value
    CD0 = cd04_sl.value
    AR = ar4_sl.value
    e_ow = 0.8
    T_avail = Ta_sl.value
    _, _, rho = isa(h)
    K = 1 / (np.pi * e_ow * AR)
    V = np.linspace(30, 350, 600)
    q = 0.5 * rho * V**2
    S = 122.6
    D_para = q * S * CD0
    D_ind = K * W**2 / (q * S)
    D_total = D_para + D_ind
    P_para = D_para * V
    P_ind = D_ind * V
    P_total = D_total * V
    V_md = np.sqrt(2 * W / (rho * S) * np.sqrt(1 / (np.pi * e_ow * AR * CD0)))
    V_mp = V_md / 3**0.25
    D_min = D_total[np.argmin(D_total)]

    with out4:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # Thrust required
        ax1.plot(V * 3.6, D_total / 1e3, 'k', lw=2.5, label='Total drag')
        ax1.plot(V * 3.6, D_para / 1e3, 'C0', lw=1.5, ls='--', label='Parasite')
        ax1.plot(V * 3.6, D_ind / 1e3, 'C1', lw=1.5, ls='--', label='Induced')
        ax1.axhline(T_avail / 1e3, color='C2', lw=2, ls='-.', label=f'T_avail = {T_avail/1e3:.0f} kN')
        ax1.axvline(V_md * 3.6, ls=':', color='gray', lw=1)
        ax1.text(V_md*3.6+5, ax1.get_ylim()[0]+5 if D_min/1e3 > 5 else 2,
                 f'V_MD = {V_md:.0f} m/s', fontsize=8, color='gray')
        # max speed
        above = np.where(D_total < T_avail)[0]
        if len(above) > 0:
            V_max = V[above[-1]]
            V_min = V[above[0]]
            ax1.plot(V_max*3.6, T_avail/1e3, 'v', color='C2', ms=10, zorder=5)
            ax1.text(V_max*3.6, T_avail/1e3+3, f'V_max={V_max:.0f} m/s',
                     fontsize=8, color='C2', ha='center')
        ax1.set_xlabel('TAS (km/h)')
        ax1.set_ylabel('Thrust / Drag (kN)')
        ax1.set_title('Thrust required vs available')
        ax1.legend(fontsize=8)
        ax1.set_xlim(0, 350 * 3.6)

        # Power required
        ax2.plot(V * 3.6, P_total / 1e6, 'k', lw=2.5, label='Total')
        ax2.plot(V * 3.6, P_para / 1e6, 'C0', lw=1.5, ls='--', label='Parasite')
        ax2.plot(V * 3.6, P_ind / 1e6, 'C1', lw=1.5, ls='--', label='Induced')
        ax2.axhline(T_avail * V_md / 1e6, color='C2', lw=1, ls=':', alpha=0.5)
        ax2.axvline(V_mp * 3.6, ls=':', color='C4', lw=1)
        ax2.text(V_mp*3.6+5, P_total.min()/1e6,
                 f'V_MP = {V_mp:.0f} m/s', fontsize=8, color='C4')
        ax2.set_xlabel('TAS (km/h)')
        ax2.set_ylabel('Power (MW)')
        ax2.set_title('Power required')
        ax2.legend(fontsize=8)
        ax2.set_xlim(0, 350 * 3.6)

        plt.tight_layout()
        plt.show()

for w in [W_sl, h4_sl, cd04_sl, ar4_sl, Ta_sl]:
    w.observe(update_perf, 'value')
display(widgets.VBox([W_sl, h4_sl, cd04_sl, ar4_sl, Ta_sl]), out4)
update_perf()

Output()

## 5 — Climb Performance

Rate of climb from excess power:

$$\text{ROC} = \frac{(T - D)\,V}{W} = \frac{P_{\text{excess}}}{W}$$

| Speed | Meaning |
|:--|:--|
| $V_x$ | Best **angle** of climb (max $\gamma$) — steepest path |
| $V_y$ | Best **rate** of climb (max ROC) — fastest altitude gain |

The **absolute ceiling** is where ROC = 0. The **service ceiling** is defined as ROC = 100 ft/min (0.508 m/s).

In [5]:
out5 = widgets.Output()

T5_sl = FloatSlider(value=120000, min=10000, max=500000, step=5000,
    description='Thrust (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
W5_sl = FloatSlider(value=700000, min=50000, max=2000000, step=50000,
    description='Weight (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
h5_sl = FloatSlider(value=0, min=0, max=12000, step=500,
    description='Altitude (m)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_climb(change=None):
    T = T5_sl.value
    W = W5_sl.value
    h = h5_sl.value
    S = 122.6
    CD0 = 0.020
    e_ow = 0.8
    AR = 9.0
    K = 1 / (np.pi * e_ow * AR)
    _, _, rho = isa(h)
    V = np.linspace(40, 300, 500)
    q = 0.5 * rho * V**2
    D = q * S * CD0 + K * W**2 / (q * S)
    ROC = (T - D) * V / W
    gamma = np.arcsin(np.clip((T - D) / W, -1, 1))
    # best ROC
    idx_vy = np.argmax(ROC)
    Vy = V[idx_vy]
    ROC_max = ROC[idx_vy]
    # best angle
    idx_vx = np.argmax(gamma)
    Vx = V[idx_vx]
    # ceiling diagram
    h_arr = np.linspace(0, 15000, 200)
    ROC_ceil = []
    for hi in h_arr:
        _, _, rho_i = isa(hi)
        # thrust lapse with altitude (simplified)
        sigma = rho_i / RHO_SL
        Ti = T * sigma
        V_md_i = np.sqrt(2*W/(rho_i*S) * np.sqrt(1/(np.pi*e_ow*AR*CD0)))
        q_i = 0.5 * rho_i * V_md_i**2
        D_i = q_i*S*CD0 + K*W**2/(q_i*S)
        roc_i = max((Ti - D_i) * V_md_i / W, 0)
        ROC_ceil.append(roc_i)
    ROC_ceil = np.array(ROC_ceil)

    with out5:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # ROC vs V
        ax1.plot(V * 3.6, ROC, 'C0', lw=2.5)
        ax1.axhline(0, color='gray', lw=0.5)
        ax1.plot(Vy*3.6, ROC_max, 'o', color='C0', ms=10, zorder=5)
        ax1.annotate(f'Vy = {Vy:.0f} m/s\nROC = {ROC_max:.1f} m/s',
                     (Vy*3.6, ROC_max), xytext=(Vy*3.6+30, ROC_max-3),
                     fontsize=9, color='C0',
                     arrowprops=dict(arrowstyle='->', color='C0'))
        ax1.plot(Vx*3.6, ROC[idx_vx], 's', color='C2', ms=10, zorder=5)
        ax1.annotate(f'Vx = {Vx:.0f} m/s\nγ = {np.rad2deg(gamma[idx_vx]):.1f}°',
                     (Vx*3.6, ROC[idx_vx]), xytext=(Vx*3.6-80, ROC[idx_vx]+2),
                     fontsize=9, color='C2',
                     arrowprops=dict(arrowstyle='->', color='C2'))
        ax1.set_xlabel('TAS (km/h)')
        ax1.set_ylabel('Rate of climb (m/s)')
        ax1.set_title(f'Climb performance at h = {h:.0f} m')
        ax1.set_xlim(0, 300*3.6)

        # ceiling diagram
        ax2.plot(ROC_ceil, h_arr / 1e3, 'C3', lw=2.5)
        ax2.axvline(0.508, ls='--', color='C1', lw=1.2, label='Service ceiling (100 ft/min)')
        # find service ceiling
        sc_idx = np.where(ROC_ceil >= 0.508)[0]
        if len(sc_idx) > 0:
            h_sc = h_arr[sc_idx[-1]] / 1e3
            ax2.plot(0.508, h_sc, 'o', color='C1', ms=8, zorder=5)
            ax2.text(1, h_sc, f'{h_sc:.1f} km', fontsize=9, color='C1')
        # absolute ceiling
        ac_idx = np.where(ROC_ceil > 0.1)[0]
        if len(ac_idx) > 0:
            h_ac = h_arr[ac_idx[-1]] / 1e3
            ax2.plot(0, h_ac, 's', color='C3', ms=8, zorder=5)
            ax2.text(0.3, h_ac, f'Abs. ceiling {h_ac:.1f} km', fontsize=9, color='C3')
        ax2.set_xlabel('Max ROC (m/s)')
        ax2.set_ylabel('Altitude (km)')
        ax2.set_title('Ceiling diagram')
        ax2.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

for w in [T5_sl, W5_sl, h5_sl]:
    w.observe(update_climb, 'value')
display(widgets.VBox([T5_sl, W5_sl, h5_sl]), out5)
update_climb()

Output()

## 6 — Glide Performance

With no thrust, the glide ratio equals $L/D$:

$$\text{Glide range} = h \times \frac{L}{D}, \qquad \text{Sink rate} = \frac{V}{L/D}$$

| Aircraft | L/D | Notes |
|:--|:--|:--|
| Space Shuttle | 4.5 | Unpowered brick |
| F-16 | 10 | Fighter |
| Boeing 737 | 17 | Transport |
| Sailplane | 40–60 | Optimized glider |
| Albatross | 20 | Nature's glider |

In [6]:
out6 = widgets.Output()

h6_sl = FloatSlider(value=10000, min=1000, max=15000, step=500,
    description='Altitude (m)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
ld6_sl = FloatSlider(value=17, min=3, max=50, step=1,
    description='L/D max', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
W6_sl = FloatSlider(value=700000, min=50000, max=2000000, step=50000,
    description='Weight (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_glide(change=None):
    h0 = h6_sl.value
    LD_max = ld6_sl.value
    W = W6_sl.value
    S = 122.6
    glide_range = h0 * LD_max
    # reference aircraft
    refs = [('Shuttle', 4.5, 'C3'), ('F-16', 10, 'C4'),
            ('B737', 17, 'C0'), ('Sailplane', 45, 'C2')]
    # speed polar
    _, _, rho = isa(h0 / 2)  # average altitude density
    CD0 = 1 / (4 * LD_max**2 * np.pi * 0.8 * 9.0)  # back-calculate
    # just use CD0 = 0.02 and back-calc e*AR for given L/D
    eAR = 1 / (4 * CD0 * LD_max**2) if LD_max > 0 else 1
    # This gives L/D_max = 0.5 * sqrt(pi*eAR/CD0)
    # Actually: (L/D)max = 1/(2*sqrt(CD0*K)) where K = 1/(pi*e*AR)
    # So K = 1/(4*CD0*LD_max^2)
    CD0 = 0.020
    K = 1 / (4 * CD0 * LD_max**2)
    V_arr = np.linspace(30, 250, 400)
    CL = 2 * W / (rho * V_arr**2 * S)
    CD = CD0 + K * CL**2
    gamma_g = -np.arctan(CD / CL)
    Vsink = -V_arr * np.sin(gamma_g)
    V_bg = V_arr[np.argmin(Vsink / V_arr)]  # best glide (min sink/V = max L/D)
    V_ms = V_arr[np.argmin(Vsink)]  # min sink

    with out6:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # glide trajectory
        for name, ld, c in refs:
            rng = h0 * ld
            ax1.plot([0, rng/1e3], [h0/1e3, 0], color=c, lw=1.5, ls='--',
                     label=f'{name} L/D={ld} → {rng/1e3:.0f} km', alpha=0.6)
        ax1.plot([0, glide_range/1e3], [h0/1e3, 0], 'k', lw=2.5,
                 label=f'Selected L/D={LD_max} → {glide_range/1e3:.0f} km')
        ax1.set_xlabel('Distance (km)')
        ax1.set_ylabel('Altitude (km)')
        ax1.set_title(f'Glide trajectories from {h0/1e3:.1f} km')
        ax1.legend(fontsize=8)
        ax1.set_ylim(0, h0/1e3 * 1.1)

        # speed polar
        ax2.plot(V_arr * 3.6, Vsink, 'C0', lw=2.5)
        ax2.plot(V_ms*3.6, Vsink.min(), 'o', color='C2', ms=10, zorder=5)
        ax2.annotate(f'Min sink\nV={V_ms:.0f} m/s\n{Vsink.min():.1f} m/s',
                     (V_ms*3.6, Vsink.min()), xytext=(V_ms*3.6-80, Vsink.min()+1),
                     fontsize=8, color='C2',
                     arrowprops=dict(arrowstyle='->', color='C2'))
        ax2.plot(V_bg*3.6, Vsink[np.argmin(Vsink/V_arr)], 's', color='C1', ms=10, zorder=5)
        ax2.annotate(f'Best glide\nV={V_bg:.0f} m/s',
                     (V_bg*3.6, Vsink[np.argmin(Vsink/V_arr)]),
                     xytext=(V_bg*3.6+30, Vsink[np.argmin(Vsink/V_arr)]+2),
                     fontsize=8, color='C1',
                     arrowprops=dict(arrowstyle='->', color='C1'))
        ax2.set_xlabel('TAS (km/h)')
        ax2.set_ylabel('Sink rate (m/s)')
        ax2.set_title('Speed polar')
        ax2.invert_yaxis()
        ax2.set_xlim(0, 250*3.6)
        plt.tight_layout()
        plt.show()

for w in [h6_sl, ld6_sl, W6_sl]:
    w.observe(update_glide, 'value')
display(widgets.VBox([h6_sl, ld6_sl, W6_sl]), out6)
update_glide()

Output()

## 7 — The V-n Diagram (Flight Envelope)

The **load factor** $n = L/W$ defines the structural demand on the aircraft.

The stall boundary (maximum achievable $n$ at each speed):

$$n_{\max} = \frac{\tfrac{1}{2}\rho V^2 S\, C_{L_{\max}}}{W}$$

**Maneuvering speed** $V_A$: the speed at which pulling $C_{L_{\max}}$ just reaches the structural limit $n_{\text{limit}}$:

$$V_A = V_{\text{stall}} \sqrt{n_{\text{limit}}}$$

| Category | $n_{+}$ | $n_{-}$ |
|:--|:--|:--|
| Normal (FAR 23) | +3.8 | −1.52 |
| Utility | +4.4 | −1.76 |
| Aerobatic | +6.0 | −3.0 |
| Transport (FAR 25) | +2.5 | −1.0 |

In [7]:
out7 = widgets.Output()

clmax7 = FloatSlider(value=1.8, min=1.0, max=3.0, step=0.1,
    description='CL_max', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
npos_sl = FloatSlider(value=3.8, min=2.0, max=9.0, step=0.1,
    description='n+ limit', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
nneg_sl = FloatSlider(value=-1.52, min=-4.0, max=-0.5, step=0.1,
    description='n− limit', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
vne_sl = FloatSlider(value=200, min=50, max=400, step=5,
    description='Vne (m/s)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
W7_sl = FloatSlider(value=12000, min=2000, max=300000, step=1000,
    description='Weight (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
S7_sl = FloatSlider(value=16.2, min=5, max=200, step=1,
    description='Wing area (m²)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_vn(change=None):
    CLmax = clmax7.value
    CLmax_neg = -0.8 * CLmax  # negative CLmax ~80% of positive
    n_pos = npos_sl.value
    n_neg = nneg_sl.value
    Vne = vne_sl.value
    W = W7_sl.value
    S = S7_sl.value
    rho = RHO_SL
    Vs = np.sqrt(2 * W / (rho * S * CLmax))
    Va = Vs * np.sqrt(n_pos)
    V = np.linspace(0, Vne * 1.1, 600)
    # stall boundaries
    n_stall_pos = 0.5 * rho * V**2 * S * CLmax / W
    n_stall_neg = 0.5 * rho * V**2 * S * CLmax_neg / W

    with out7:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(12, 6))
        # flight envelope
        # positive stall curve up to Va
        mask_pre_va = V <= Va
        mask_post_va = (V >= Va) & (V <= Vne)
        ax.plot(V[mask_pre_va] * 3.6, np.clip(n_stall_pos[mask_pre_va], 0, n_pos),
                'C0', lw=2.5)
        ax.plot(V[mask_post_va] * 3.6, np.full(mask_post_va.sum(), n_pos),
                'C0', lw=2.5)
        # Vne vertical
        ax.plot([Vne*3.6, Vne*3.6], [n_neg, n_pos], 'C3', lw=2.5)
        # negative stall curve
        Vs_neg = np.sqrt(2 * W / (rho * S * abs(CLmax_neg)))
        Va_neg = Vs_neg * np.sqrt(abs(n_neg))
        mask_pre_va_neg = V <= Va_neg
        mask_post_va_neg = (V >= Va_neg) & (V <= Vne)
        ax.plot(V[mask_pre_va_neg] * 3.6, np.clip(n_stall_neg[mask_pre_va_neg], n_neg, 0),
                'C0', lw=2.5)
        ax.plot(V[mask_post_va_neg] * 3.6, np.full(mask_post_va_neg.sum(), n_neg),
                'C0', lw=2.5)
        # fill envelope
        V_env = np.concatenate([
            V[mask_pre_va], V[mask_post_va], [Vne],
            [Vne], V[mask_post_va_neg][::-1], V[mask_pre_va_neg][::-1]
        ])
        n_env = np.concatenate([
            np.clip(n_stall_pos[mask_pre_va], 0, n_pos),
            np.full(mask_post_va.sum(), n_pos), [n_pos],
            [n_neg], np.full(mask_post_va_neg.sum(), n_neg),
            np.clip(n_stall_neg[mask_pre_va_neg], n_neg, 0)[::-1]
        ])
        ax.fill(V_env * 3.6, n_env, alpha=0.1, color='C0')
        # reference lines
        ax.axhline(1, ls=':', color='gray', lw=0.8)
        ax.axhline(0, ls='-', color='gray', lw=0.5)
        # annotations
        ax.axvline(Vs*3.6, ls='--', color='C1', lw=1, alpha=0.6)
        ax.text(Vs*3.6, n_pos+0.15, f'Vs={Vs:.0f} m/s', fontsize=8, color='C1', ha='center')
        ax.axvline(Va*3.6, ls='--', color='C2', lw=1, alpha=0.6)
        ax.text(Va*3.6, n_pos+0.15, f'Va={Va:.0f} m/s', fontsize=8, color='C2', ha='center')
        ax.text(Vne*3.6+5, 0, f'Vne={Vne:.0f} m/s', fontsize=8, color='C3', rotation=90)
        # structural limit labels
        ax.text(Vne*3.6*0.7, n_pos+0.15, f'n+ = {n_pos:.1f}g', fontsize=9,
                color='C0', fontweight='bold')
        ax.text(Vne*3.6*0.7, n_neg-0.25, f'n− = {n_neg:.1f}g', fontsize=9,
                color='C0', fontweight='bold')
        ax.set_xlabel('Equivalent airspeed (km/h)')
        ax.set_ylabel('Load factor n (g)')
        ax.set_title('V-n Diagram — Flight Envelope', fontsize=12)
        ax.set_xlim(0, Vne * 1.15 * 3.6)
        ax.set_ylim(n_neg - 0.8, n_pos + 0.8)
        plt.tight_layout()
        plt.show()

for w in [clmax7, npos_sl, nneg_sl, vne_sl, W7_sl, S7_sl]:
    w.observe(update_vn, 'value')
display(widgets.VBox([clmax7, npos_sl, nneg_sl, vne_sl, W7_sl, S7_sl]), out7)
update_vn()

Output()

## 8 — Turning Flight

In a coordinated banked turn, the lift vector tilts inward. The load factor and turn geometry:

$$n = \frac{1}{\cos\phi}, \qquad R = \frac{V^2}{g\,\tan\phi}, \qquad \omega = \frac{g\,\tan\phi}{V}$$

| Symbol | Meaning |
|:--|:--|
| $\phi$ | Bank angle |
| $R$ | Turn radius |
| $\omega$ | Turn rate (°/s) |
| $n$ | Load factor |

Stall speed in a turn: $V_{s,\text{turn}} = V_s\sqrt{n}$. A **standard rate turn** is 3°/s (2 min for 360°).

In [8]:
out8 = widgets.Output()

phi_sl = FloatSlider(value=45, min=5, max=80, step=1,
    description='Bank angle (°)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
v8_sl = FloatSlider(value=80, min=30, max=250, step=5,
    description='TAS (m/s)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_turn(change=None):
    phi = np.deg2rad(phi_sl.value)
    V = v8_sl.value
    n = 1 / np.cos(phi)
    R = V**2 / (G0 * np.tan(phi))
    omega = np.rad2deg(G0 * np.tan(phi) / V)  # deg/s
    # sweep bank angle
    phi_arr = np.linspace(5, 80, 300)
    phi_rad = np.deg2rad(phi_arr)
    n_arr = 1 / np.cos(phi_rad)
    R_arr = V**2 / (G0 * np.tan(phi_rad))
    omega_arr = np.rad2deg(G0 * np.tan(phi_rad) / V)

    with out8:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
        # force diagram
        ax1.set_xlim(-2, 2)
        ax1.set_ylim(-1.5, 2)
        ax1.set_aspect('equal')
        ax1.axis('off')
        ax1.set_title(f'Banked turn — φ = {phi_sl.value:.0f}°  n = {n:.2f}g', fontsize=10)
        # aircraft cross-section (tilted)
        wing_len = 1.2
        cx, cy = 0, 0
        dx = wing_len * np.cos(phi)
        dy = wing_len * np.sin(phi)
        ax1.plot([cx-dx, cx+dx], [cy-dy, cy+dy], 'k', lw=4)
        ax1.plot(cx, cy, 'ko', ms=8, zorder=5)
        # weight (down)
        W_arrow = 1.0
        ax1.annotate('', xy=(cx, cy - W_arrow), xytext=(cx, cy),
                     arrowprops=dict(arrowstyle='->', color='C3', lw=2.5))
        ax1.text(cx + 0.15, cy - W_arrow/2, 'W', fontsize=12, color='C3', fontweight='bold')
        # lift (perpendicular to wing, upward)
        L_arrow = W_arrow * n
        L_dx = -L_arrow * np.sin(phi)
        L_dy = L_arrow * np.cos(phi)
        ax1.annotate('', xy=(cx + L_dx, cy + L_dy), xytext=(cx, cy),
                     arrowprops=dict(arrowstyle='->', color='C0', lw=2.5))
        ax1.text(cx + L_dx - 0.3, cy + L_dy/2, f'L = {n:.1f}W',
                 fontsize=11, color='C0', fontweight='bold')
        # centripetal (horizontal, inward)
        Fc_arrow = W_arrow * np.tan(phi)
        ax1.annotate('', xy=(cx - Fc_arrow, cy), xytext=(cx, cy),
                     arrowprops=dict(arrowstyle='->', color='C1', lw=2))
        ax1.text(cx - Fc_arrow/2, cy + 0.15, 'Fc', fontsize=11, color='C1', fontweight='bold')
        # info text
        ax1.text(0, -1.3, f'R = {R:.0f} m ({R/1e3:.2f} km)\n'
                 f'ω = {omega:.1f}°/s\n'
                 f'Time for 360° = {360/omega:.1f} s',
                 ha='center', fontsize=10, fontfamily='monospace',
                 bbox=dict(fc='lightyellow', ec='gray', pad=5))

        # turn radius and rate vs bank
        ax2a = ax2
        ax2a.semilogy(phi_arr, R_arr, 'C0', lw=2, label='Turn radius')
        ax2a.axhline(R, ls=':', color='C0', lw=1, alpha=0.5)
        ax2a.plot(phi_sl.value, R, 'o', color='C0', ms=8, zorder=5)
        ax2a.set_xlabel('Bank angle (°)')
        ax2a.set_ylabel('Turn radius (m)', color='C0')
        ax2a.tick_params(axis='y', labelcolor='C0')
        ax2b = ax2a.twinx()
        ax2b.plot(phi_arr, omega_arr, 'C3', lw=2, label='Turn rate')
        ax2b.axhline(3, ls='--', color='gray', lw=1, alpha=0.5)
        ax2b.text(10, 3.3, 'Standard rate (3°/s)', fontsize=8, color='gray')
        ax2b.plot(phi_sl.value, omega, 's', color='C3', ms=8, zorder=5)
        ax2b.set_ylabel('Turn rate (°/s)', color='C3')
        ax2b.tick_params(axis='y', labelcolor='C3')
        ax2a.set_title(f'Turn geometry at V = {V:.0f} m/s ({V*3.6:.0f} km/h)', fontsize=10)
        fig.legend(loc='upper center', ncol=2, fontsize=9, bbox_to_anchor=(0.75, 0.95))
        plt.tight_layout()
        plt.show()

for w in [phi_sl, v8_sl]:
    w.observe(update_turn, 'value')
display(widgets.VBox([phi_sl, v8_sl]), out8)
update_turn()

Output()

## 9 — Takeoff Performance

The ground roll distance integrates the equation of motion:

$$m\frac{dV}{dt} = T - D - \mu_r(W - L)$$

| Speed | Meaning |
|:--|:--|
| $V_1$ | Decision speed — go/no-go |
| $V_R$ | Rotation speed — pull back on stick |
| $V_2$ | Takeoff safety speed (≥ 1.2 $V_s$) |

**Density altitude** increases takeoff distance: thinner air → less lift and less thrust.

In [9]:
out9 = widgets.Output()

W9_sl = FloatSlider(value=70000, min=10000, max=300000, step=5000,
    description='Weight (kg)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
T9_sl = FloatSlider(value=120000, min=20000, max=500000, step=5000,
    description='Thrust (N)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
elev_sl = FloatSlider(value=0, min=0, max=3000, step=100,
    description='Elevation (m)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
hw_sl = FloatSlider(value=0, min=-10, max=30, step=1,
    description='Headwind (m/s)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_takeoff(change=None):
    mass = W9_sl.value
    W = mass * G0
    T_eng = T9_sl.value
    h_elev = elev_sl.value
    Vw = hw_sl.value
    S = 122.6
    CL_ground = 0.8
    CL_rotate = 1.6
    CD_ground = 0.040
    mu_r = 0.02  # rolling friction
    _, _, rho = isa(h_elev)
    Vs = np.sqrt(2 * W / (rho * S * CL_rotate))
    Vr = 1.1 * Vs  # rotation speed
    V2 = 1.2 * Vs  # safety speed
    V1 = 0.95 * Vr  # decision speed (simplified)
    # integrate ground roll
    dt = 0.1
    V_list, s_list = [0], [0]
    V, s = 0, 0
    F_thrust, F_drag, F_fric, F_net = [], [], [], []
    V_force = []
    while V < V2 and s < 5000:
        Va = V + Vw  # airspeed
        q = 0.5 * rho * max(Va, 0.1)**2
        L = q * S * CL_ground
        D = q * S * CD_ground
        friction = mu_r * max(W - L, 0)
        F = T_eng - D - friction
        F_thrust.append(T_eng)
        F_drag.append(D)
        F_fric.append(friction)
        F_net.append(F)
        V_force.append(V)
        a = F / mass
        V += a * dt
        s += V * dt
        V_list.append(V)
        s_list.append(s)
    V_arr = np.array(V_list)
    s_arr = np.array(s_list)
    V_f = np.array(V_force)
    # find ground roll distance
    idx_vr = np.argmin(np.abs(V_arr - Vr))
    s_gr = s_arr[idx_vr] if idx_vr < len(s_arr) else s_arr[-1]

    with out9:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # V vs distance
        ax1.plot(s_arr, V_arr, 'C0', lw=2.5)
        ax1.axhline(V1, ls=':', color='C1', lw=1.2)
        ax1.text(s_arr[-1]*0.05, V1+1, f'V1 = {V1:.0f} m/s', fontsize=8, color='C1')
        ax1.axhline(Vr, ls='--', color='C2', lw=1.2)
        ax1.text(s_arr[-1]*0.05, Vr+1, f'VR = {Vr:.0f} m/s', fontsize=8, color='C2')
        ax1.axhline(V2, ls='-.', color='C4', lw=1.2)
        ax1.text(s_arr[-1]*0.05, V2+1, f'V2 = {V2:.0f} m/s', fontsize=8, color='C4')
        ax1.axvline(s_gr, ls=':', color='gray', lw=1)
        ax1.text(s_gr+20, V_arr.max()*0.3, f'Ground roll\n{s_gr:.0f} m', fontsize=9,
                 color='gray')
        ax1.set_xlabel('Distance (m)')
        ax1.set_ylabel('Ground speed (m/s)')
        ax1.set_title(f'Takeoff roll — elevation {h_elev:.0f} m  ρ = {rho:.3f} kg/m³')

        # forces vs V
        ax2.plot(np.array(V_f) * 3.6, np.array(F_thrust) / 1e3, 'C2', lw=1.5, label='Thrust')
        ax2.plot(np.array(V_f) * 3.6, np.array(F_drag) / 1e3, 'C1', lw=1.5, label='Drag')
        ax2.plot(np.array(V_f) * 3.6, np.array(F_fric) / 1e3, 'C4', lw=1.5, label='Friction')
        ax2.plot(np.array(V_f) * 3.6, np.array(F_net) / 1e3, 'k', lw=2.5, label='Net force')
        ax2.set_xlabel('Ground speed (km/h)')
        ax2.set_ylabel('Force (kN)')
        ax2.set_title('Forces during takeoff roll')
        ax2.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

for w in [W9_sl, T9_sl, elev_sl, hw_sl]:
    w.observe(update_takeoff, 'value')
display(widgets.VBox([W9_sl, T9_sl, elev_sl, hw_sl]), out9)
update_takeoff()

Output()

## 10 — Longitudinal Static Stability

An aircraft is statically stable in pitch if a nose-up disturbance produces a **nose-down restoring moment**:

$$\frac{dC_m}{d\alpha} < 0 \qquad \text{(negative slope)}$$

The **neutral point** (NP) is the aft-most CG position for static stability. The **static margin**:

$$\text{SM} = \frac{x_{NP} - x_{CG}}{\bar{c}} \qquad \text{(positive = stable)}$$

Typical static margins: fighters 5–10%, transports 10–15%, trainers 15–25%.

The pitching moment coefficient:

$$C_m = C_{m_0} + C_{m_\alpha}\,\alpha, \qquad C_{m_\alpha} = C_{L_\alpha}\left(\frac{x_{CG} - x_{NP}}{\bar{c}}\right)$$

In [ ]:
out10 = widgets.Output()

cg_sl = FloatSlider(value=25, min=10, max=50, step=1,
    description='CG (% MAC)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
np_sl = FloatSlider(value=35, min=20, max=55, step=1,
    description='NP (% MAC)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
cm0_sl = FloatSlider(value=0.08, min=-0.05, max=0.20, step=0.01,
    description='Cm0', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_stability(change=None):
    x_cg = cg_sl.value / 100
    x_np = np_sl.value / 100
    Cm0 = cm0_sl.value
    SM = x_np - x_cg  # static margin (fraction of MAC)
    CLa = 5.0  # per radian
    Cma = CLa * (x_cg - x_np)  # negative = stable
    alpha = np.linspace(-5, 20, 300)
    alpha_rad = np.deg2rad(alpha)
    Cm = Cm0 + Cma * alpha_rad
    # trim angle (Cm = 0)
    if abs(Cma) > 0.01:
        alpha_trim = -Cm0 / Cma  # radians
        alpha_trim_deg = np.rad2deg(alpha_trim)
    else:
        alpha_trim_deg = None
    stable = SM > 0

    with out10:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        # Cm vs alpha
        color = 'C2' if stable else 'C3'
        ax1.plot(alpha, Cm, color=color, lw=2.5)
        ax1.axhline(0, color='gray', lw=0.8)
        ax1.axvline(0, color='gray', lw=0.3)
        if alpha_trim_deg is not None and -5 < alpha_trim_deg < 20:
            ax1.plot(alpha_trim_deg, 0, 'o', color=color, ms=10, zorder=5)
            ax1.annotate(f'Trim α = {alpha_trim_deg:.1f}°',
                         (alpha_trim_deg, 0), xytext=(alpha_trim_deg+3, 0.05),
                         fontsize=9, color=color,
                         arrowprops=dict(arrowstyle='->', color=color))
        status = 'STABLE' if stable else 'UNSTABLE'
        ax1.text(0.95, 0.95, f'{status}\nSM = {SM*100:.0f}% MAC\n'
                 f'Cmα = {Cma:.3f} /rad',
                 transform=ax1.transAxes, ha='right', va='top', fontsize=10,
                 color=color, fontweight='bold',
                 bbox=dict(fc='white', ec=color, lw=2, pad=5))
        ax1.set_xlabel('Angle of attack α (°)')
        ax1.set_ylabel('Cm (pitching moment coeff.)')
        ax1.set_title('Longitudinal stability')

        # aircraft side view with CG, NP
        ax2.set_xlim(-0.5, 1.5)
        ax2.set_ylim(-0.5, 0.8)
        ax2.set_aspect('equal')
        ax2.axis('off')
        ax2.set_title('CG and Neutral Point on MAC', fontsize=10)
        # fuselage
        fuse = plt.Polygon([[0, -0.05], [1.3, -0.03], [1.3, 0.03], [0, 0.05]],
                           fc='#bdc3c7', ec='k', lw=1.5)
        ax2.add_patch(fuse)
        # MAC line
        mac_x0, mac_x1 = 0.2, 0.8
        ax2.plot([mac_x0, mac_x1], [-0.15, -0.15], 'k', lw=3)
        ax2.text((mac_x0+mac_x1)/2, -0.25, 'MAC', ha='center', fontsize=9)
        # CG
        cg_x = mac_x0 + x_cg * (mac_x1 - mac_x0)
        ax2.plot(cg_x, 0, 'o', color='C0', ms=14, zorder=5)
        ax2.text(cg_x, 0.12, f'CG ({cg_sl.value:.0f}%)', ha='center',
                 fontsize=9, color='C0', fontweight='bold')
        # NP
        np_x = mac_x0 + x_np * (mac_x1 - mac_x0)
        ax2.plot(np_x, 0, 's', color=color, ms=12, zorder=5)
        ax2.text(np_x, -0.08, f'NP ({np_sl.value:.0f}%)', ha='center',
                 fontsize=9, color=color, fontweight='bold')
        # static margin arrow
        if abs(cg_x - np_x) > 0.01:
            ax2.annotate('', xy=(np_x, 0.25), xytext=(cg_x, 0.25),
                         arrowprops=dict(arrowstyle='<->', color=color, lw=2))
            ax2.text((cg_x+np_x)/2, 0.30, f'SM = {SM*100:.0f}%',
                     ha='center', fontsize=10, color=color, fontweight='bold')
        plt.tight_layout()
        plt.show()

for w in [cg_sl, np_sl, cm0_sl]:
    w.observe(update_stability, 'value')
display(widgets.VBox([cg_sl, np_sl, cm0_sl]), out10)
update_stability()

## 11 — Dynamic Stability: Longitudinal Modes

An aircraft disturbed in pitch exhibits two natural modes:

| Mode | Period | Damping | What oscillates |
|:--|:--|:--|:--|
| **Short period** | 1–5 s | Well damped | Pitch rate, angle of attack |
| **Phugoid** | 30–120 s | Lightly damped | Airspeed ↔ altitude (constant α) |

**Short period** approximation:
$$\omega_{sp} \approx \sqrt{-\frac{\rho V S \bar{c}}{2I_y}\,C_{m_\alpha}}, \qquad \zeta_{sp} \approx \frac{-C_{m_q}}{2\omega_{sp}}$$

**Phugoid** approximation:
$$\omega_{ph} \approx \frac{g\sqrt{2}}{V}, \qquad \zeta_{ph} \approx \frac{1}{\sqrt{2}}\frac{C_D}{C_L} = \frac{1}{\sqrt{2}\,(L/D)}$$

In [ ]:
out11 = widgets.Output()

v11_sl = FloatSlider(value=80, min=40, max=250, step=5,
    description='TAS (m/s)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
sm11_sl = FloatSlider(value=10, min=1, max=30, step=1,
    description='Static margin (%)', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
ld11_sl = FloatSlider(value=15, min=5, max=40, step=1,
    description='L/D', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)
zeta_sp_sl = FloatSlider(value=0.3, min=0.05, max=1.0, step=0.05,
    description='ζ_sp', style={'description_width':'110px'},
    layout=widgets.Layout(width='560px'), continuous_update=False)

def update_dynamic(change=None):
    V = v11_sl.value
    SM = sm11_sl.value / 100
    LD = ld11_sl.value
    zeta_sp = zeta_sp_sl.value
    # short period
    omega_sp = 2.0 * np.sqrt(SM * 10)  # simplified: higher SM → higher freq
    # phugoid
    omega_ph = G0 * np.sqrt(2) / V
    zeta_ph = 1 / (np.sqrt(2) * LD)
    # time responses
    t_sp = np.linspace(0, 10, 1000)
    t_ph = np.linspace(0, 300, 2000)
    # damped oscillation: x(t) = exp(-zeta*omega*t) * cos(omega_d*t)
    omega_d_sp = omega_sp * np.sqrt(1 - min(zeta_sp**2, 0.99))
    omega_d_ph = omega_ph * np.sqrt(1 - min(zeta_ph**2, 0.99))
    q_sp = np.exp(-zeta_sp * omega_sp * t_sp) * np.cos(omega_d_sp * t_sp)
    # phugoid: velocity and altitude oscillation (opposite phase)
    dV_ph = np.exp(-zeta_ph * omega_ph * t_ph) * np.cos(omega_d_ph * t_ph)
    dh_ph = -np.exp(-zeta_ph * omega_ph * t_ph) * np.cos(omega_d_ph * t_ph - np.pi/2)
    T_sp = 2 * np.pi / omega_sp if omega_sp > 0 else 0
    T_ph = 2 * np.pi / omega_ph if omega_ph > 0 else 0
    # eigenvalues
    sp_real = -zeta_sp * omega_sp
    sp_imag = omega_d_sp
    ph_real = -zeta_ph * omega_ph
    ph_imag = omega_d_ph

    with out11:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
        # short period
        ax = axes[0]
        ax.plot(t_sp, q_sp, 'C0', lw=2)
        ax.plot(t_sp, np.exp(-zeta_sp * omega_sp * t_sp), 'C0', ls=':', lw=1, alpha=0.5)
        ax.plot(t_sp, -np.exp(-zeta_sp * omega_sp * t_sp), 'C0', ls=':', lw=1, alpha=0.5)
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Pitch rate q (normalized)')
        ax.set_title(f'Short period\nT = {T_sp:.2f} s  ζ = {zeta_sp:.2f}', fontsize=10)
        ax.set_xlim(0, 10)

        # phugoid
        ax = axes[1]
        ax.plot(t_ph, dV_ph, 'C0', lw=2, label='ΔV/V₀')
        ax.plot(t_ph, dh_ph, 'C3', lw=2, label='Δh/h₀')
        ax.plot(t_ph, np.exp(-zeta_ph * omega_ph * t_ph), 'k', ls=':', lw=1, alpha=0.3)
        ax.plot(t_ph, -np.exp(-zeta_ph * omega_ph * t_ph), 'k', ls=':', lw=1, alpha=0.3)
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Perturbation (normalized)')
        ax.set_title(f'Phugoid\nT = {T_ph:.1f} s  ζ = {zeta_ph:.3f}', fontsize=10)
        ax.legend(fontsize=8)
        ax.set_xlim(0, 300)

        # eigenvalue plot
        ax = axes[2]
        ax.axhline(0, color='gray', lw=0.5)
        ax.axvline(0, color='gray', lw=0.5)
        # short period eigenvalues
        ax.plot(sp_real, sp_imag, 'x', color='C0', ms=12, mew=2.5, label='Short period')
        ax.plot(sp_real, -sp_imag, 'x', color='C0', ms=12, mew=2.5)
        # phugoid eigenvalues
        ax.plot(ph_real, ph_imag, 'x', color='C3', ms=12, mew=2.5, label='Phugoid')
        ax.plot(ph_real, -ph_imag, 'x', color='C3', ms=12, mew=2.5)
        # stability boundary
        ax.fill_betweenx([-omega_sp*1.5, omega_sp*1.5], 0, omega_sp*0.5,
                         alpha=0.05, color='C3')
        ax.text(0.02, omega_sp*1.2, 'Unstable', fontsize=8, color='C3')
        ax.set_xlabel('Real (1/s)')
        ax.set_ylabel('Imaginary (rad/s)')
        ax.set_title('Eigenvalue map (s-plane)', fontsize=10)
        ax.legend(fontsize=8)
        ax.set_aspect('equal')
        max_val = max(omega_sp, omega_ph) * 1.5
        ax.set_xlim(-max_val, max_val * 0.3)
        ax.set_ylim(-max_val, max_val)
        plt.tight_layout()
        plt.show()

for w in [v11_sl, sm11_sl, ld11_sl, zeta_sp_sl]:
    w.observe(update_dynamic, 'value')
display(widgets.VBox([v11_sl, sm11_sl, ld11_sl, zeta_sp_sl]), out11)
update_dynamic()